# Silver to Gold
- Transformação da camada Silver para a Gold
- Cria o modelo dimensional, tabelas fato, dimensões e bridges, além da tabela de contexto para GenAI e das consultas analíticas do projeto.

##1. Configurações Iniciais


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SILVER = "silver"
GOLD = "gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}"
)

print(f"Schema {CATALOG}.{GOLD} disponível.")

Schema workspace.gold disponível.


In [0]:
def save_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{CATALOG}.{GOLD}.{table_name}"
        )
    )

    print(
        f"{CATALOG}.{GOLD}.{table_name} criada com sucesso."
    )

####Validação prévia da Silver
- Garante que as tabelas principais possuem um registro por filme


In [0]:
def check_duplicates(df, columns, name):
    duplicates = (
        df
        .groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicates.limit(1).count() > 0:
        print(
            f"ATENÇÃO: duplicidades encontradas em {name}"
        )

        display(duplicates)

    else:
        print(
            f"OK - {name} sem duplicidades em {columns}"
        )

In [0]:
check_duplicates(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_info_filmes"
    ),
    ["id_filme"],
    "tb_info_filmes"
)

check_duplicates(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_financeiro_filmes"
    ),
    ["id_filme"],
    "tb_financeiro_filmes"
)

check_duplicates(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_metricas_engajamento"
    ),
    ["id_filme"],
    "tb_metricas_engajamento"
)

OK - tb_info_filmes sem duplicidades em ['id_filme']
OK - tb_financeiro_filmes sem duplicidades em ['id_filme']
OK - tb_metricas_engajamento sem duplicidades em ['id_filme']


##2.1 gold.dim_movies
- Armazena metadados de cada filme

##### Gerar surrogate key

In [0]:
df_info = spark.table(
    f"{CATALOG}.{SILVER}.tb_info_filmes"
)

df_movies_base = (
    df_info
    .filter(
        F.col("id_filme").isNotNull()
    )
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

In [0]:
w_movie = Window.orderBy(
    F.col("id_filme")
)

df_dim_movies = (
    df_movies_base
    .withColumn(
        "sk_movie_id",
        F.row_number()
        .over(w_movie)
        .cast("bigint")
    )
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string")
    )
)

save_gold(
    df_dim_movies,
    "dim_movies"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.dim_movies criada com sucesso.


In [0]:
display(
    spark.table(
        "workspace.gold.dim_movies"
    ).limit(20)
)

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,null,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,null,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."



## 3. gold.dim_genres
- Catalogo único e deduplicado dos gêneros


In [0]:
df_generos_silver = spark.table(
    f"{CATALOG}.{SILVER}.tb_generos"
)

df_generos_base = (
    df_generos_silver
    .filter(
        F.col("nome_genero").isNotNull()
    )
    .select("nome_genero")
    .distinct()
)

w_genre = Window.orderBy(
    F.col("nome_genero")
)

df_dim_genres = (
    df_generos_base
    .withColumn(
        "sk_genre_id",
        F.row_number()
        .over(w_genre)
        .cast("bigint")
    )
    .select(
        "sk_genre_id",
        F.col("nome_genero").cast("string")
    )
)

save_gold(
    df_dim_genres,
    "dim_genres"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.dim_genres criada com sucesso.


##4. gold.dim_people
- Todas as pessoas envolvidas na obra

In [0]:
df_entidades = spark.table(
    f"{CATALOG}.{SILVER}.tb_pessoas_empresas"
)

df_people_base = (
    df_entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
    .filter(
        F.col("nome_entidade").isNotNull()
    )
    .select(
        F.col("nome_entidade").alias(
            "nome_pessoa"
        ),
        F.col("tipo_entidade").alias(
            "tipo_pessoa"
        )
    )
    .distinct()
)

#####Deduplicação pela combinação (nome_pessoa + tipo_pessoa)

In [0]:
w_person = Window.orderBy(
    "nome_pessoa",
    "tipo_pessoa"
)

df_dim_people = (
    df_people_base
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(w_person)
        .cast("bigint")
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)

save_gold(
    df_dim_people,
    "dim_people"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.dim_people criada com sucesso.


## 5. gold.dim_companies
- Catalogo único de produtoras

In [0]:
df_companies_base = (
    df_entidades
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
    .filter(
        F.col("nome_entidade").isNotNull()
    )
    .select(
        F.col("nome_entidade").alias(
            "nome_produtora"
        )
    )
    .distinct()
)

w_company = Window.orderBy(
    "nome_produtora"
)

df_dim_companies = (
    df_companies_base
    .withColumn(
        "sk_company_id",
        F.row_number()
        .over(w_company)
        .cast("bigint")
    )
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)

save_gold(
    df_dim_companies,
    "dim_companies"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.dim_companies criada com sucesso.


##6. gold.bridge_movie_genre
 - Evita multiplicar os registros da tabela

In [0]:
df_bridge_movie_genre = (
    df_generos_silver.alias("s")
    .join(
        df_dim_movies.alias("m"),
        F.col("s.id_filme")
        == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_genres.alias("g"),
        F.col("s.nome_genero")
        == F.col("g.nome_genero"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("g.sk_genre_id")
        .cast("bigint")
        .alias("sk_genre_id")
    )
    .dropDuplicates()
)

save_gold(
    df_bridge_movie_genre,
    "bridge_movie_genre"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.bridge_movie_genre criada com sucesso.


## 7.gold.bridge_movie_person

In [0]:
df_people_silver = (
    df_entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
)

In [0]:
df_bridge_movie_person = (
    df_people_silver.alias("s")
    .join(
        df_dim_movies.alias("m"),
        F.col("s.id_filme")
        == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_people.alias("p"),
        (
            F.col("s.nome_entidade")
            == F.col("p.nome_pessoa")
        )
        &
        (
            F.col("s.tipo_entidade")
            == F.col("p.tipo_pessoa")
        ),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("p.sk_person_id")
        .cast("bigint")
        .alias("sk_person_id")
    )
    .dropDuplicates()
)

save_gold(
    df_bridge_movie_person,
    "bridge_movie_person"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.bridge_movie_person criada com sucesso.


## 8. gold.bridge_movie_company

In [0]:
df_companies_silver = (
    df_entidades
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
)

In [0]:
df_bridge_movie_company = (
    df_companies_silver.alias("s")
    .join(
        df_dim_movies.alias("m"),
        F.col("s.id_filme")
        == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_companies.alias("c"),
        F.col("s.nome_entidade")
        == F.col("c.nome_produtora"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("c.sk_company_id")
        .cast("bigint")
        .alias("sk_company_id")
    )
    .dropDuplicates()
)

save_gold(
    df_bridge_movie_company,
    "bridge_movie_company"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.bridge_movie_company criada com sucesso.


## 9. gold.dim_reviews


In [0]:
#lê a silver
df_reviews = spark.table(
    f"{CATALOG}.{SILVER}.tb_avaliacoes_usuarios"
)

In [0]:
df_reviews_agg = (
    df_reviews
    .groupBy("id_filme")
    .agg(
        F.count("*")
        .cast("int")
        .alias(
            "qtd_avaliacoes_usuarios"
        ),

        F.round(
            F.avg("nota_usuario"),
            2
        )
        .cast("double")
        .alias(
            "nota_media_usuarios"
        )
    )
)

In [0]:
df_reviews_with_movie = (
    df_reviews_agg.alias("r")
    .join(
        df_dim_movies.alias("m"),
        F.col("r.id_filme")
        == F.col("m.id_filme"),
        "inner"
    )
    .select(
        "m.sk_movie_id",
        "r.qtd_avaliacoes_usuarios",
        "r.nota_media_usuarios"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#Surrogate key
w_review = Window.orderBy(
    "sk_movie_id"
)

df_dim_reviews = (
    df_reviews_with_movie
    .withColumn(
        "sk_review_id",
        F.row_number()
        .over(w_review)
        .cast("bigint")
    )
    .select(
        "sk_review_id",
        F.col("sk_movie_id")
        .cast("bigint"),

        F.col("qtd_avaliacoes_usuarios")
        .cast("int"),

        F.col("nota_media_usuarios")
        .cast("double")
    )
)

save_gold(
    df_dim_reviews,
    "dim_reviews"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.dim_reviews criada com sucesso.


## 10. gold.fact_movies_performance

In [0]:
df_financeiro = spark.table(
    f"{CATALOG}.{SILVER}.tb_financeiro_filmes"
)

df_metricas = spark.table(
    f"{CATALOG}.{SILVER}.tb_metricas_engajamento"
)

In [0]:
df_fact_movies = (
    df_dim_movies.alias("m")

    # A fato deve possuir apenas filmes lançados.
    .filter(
        F.col("m.status_filme") == "Lançado"
    )

    .join(
        df_financeiro.alias("f"),
        F.col("m.id_filme")
        == F.col("f.id_filme"),
        "left"
    )

    .join(
        df_metricas.alias("e"),
        F.col("m.id_filme")
        == F.col("e.id_filme"),
        "left"
    )

    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("f.orcamento_usd")
        .cast("decimal(18,2)")
        .alias("orcamento_usd"),

        F.col("f.receita_usd")
        .cast("decimal(18,2)")
        .alias("receita_usd"),

        F.col("f.lucro_usd")
        .cast("decimal(18,2)")
        .alias("lucro_usd"),

        F.col("f.orcamento_brl")
        .cast("decimal(18,2)")
        .alias("orcamento_brl"),

        F.col("f.receita_brl")
        .cast("decimal(18,2)")
        .alias("receita_brl"),

        F.col("f.lucro_brl")
        .cast("decimal(18,2)")
        .alias("lucro_brl"),

        F.col("e.popularidade")
        .cast("double")
        .alias("popularidade"),

        F.col("e.nota_media_tmdb")
        .cast("double")
        .alias("nota_media_tmdb"),

        F.col("e.qtd_votos_tmdb")
        .cast("int")
        .alias("qtd_votos_tmdb"),

        F.col("e.nota_media_imdb")
        .cast("double")
        .alias("nota_media_imdb"),

        F.col("e.qtd_votos_imdb")
        .cast("int")
        .alias("qtd_votos_imdb")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
save_gold(
    df_fact_movies,
    "fact_movies_performance"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.fact_movies_performance criada com sucesso.


#### Validação


In [0]:
df_fact_validation = (
    spark.table(
        "workspace.gold.fact_movies_performance"
    )
    .groupBy("sk_movie_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Filmes duplicados na fato:",
    df_fact_validation.count()
)

Filmes duplicados na fato: 0



## 11. gold_genai_movies_context


##### Atores por filme

In [0]:
df_atores_filme = (
    df_bridge_movie_person.alias("b")
    .join(
        df_dim_people.alias("p"),
        F.col("b.sk_person_id")
        == F.col("p.sk_person_id"),
        "inner"
    )
    .filter(
        F.col("p.tipo_pessoa") == "Ator"
    )
    .groupBy(
        "b.sk_movie_id"
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set(
                    "p.nome_pessoa"
                )
            )
        ).alias("atores")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##### Diretores por filme

In [0]:
df_diretores_filme = (
    df_bridge_movie_person.alias("b")
    .join(
        df_dim_people.alias("p"),
        F.col("b.sk_person_id")
        == F.col("p.sk_person_id"),
        "inner"
    )
    .filter(
        F.col("p.tipo_pessoa") == "Diretor"
    )
    .groupBy(
        "b.sk_movie_id"
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set(
                    "p.nome_pessoa"
                )
            )
        ).alias("diretores")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



## 12. Base do Contexto


In [0]:
df_context_base = (
    df_fact_movies.alias("f")

    .join(
        df_dim_movies.alias("m"),
        F.col("f.sk_movie_id")
        == F.col("m.sk_movie_id"),
        "inner"
    )

    .join(
        df_atores_filme.alias("a"),
        F.col("f.sk_movie_id")
        == F.col("a.sk_movie_id"),
        "left"
    )

    .join(
        df_diretores_filme.alias("d"),
        F.col("f.sk_movie_id")
        == F.col("d.sk_movie_id"),
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



##### Tratamento dos NULLs


In [0]:
df_context_base = (
    df_context_base

    .withColumn(
        "_titulo",
        F.coalesce(
            F.col("m.titulo"),
            F.lit("Título não informado")
        )
    )

    .withColumn(
        "_ano",
        F.coalesce(
            F.col("m.ano_lancamento")
            .cast("string"),
            F.lit("ano não informado")
        )
    )

    .withColumn(
        "_atores",
        F.when(
            F.col("a.atores").isNull()
            | (F.length(F.trim(F.col("a.atores"))) == 0),
            F.lit("elenco não informado")
        )
        .otherwise(
            F.col("a.atores")
        )
    )

    .withColumn(
        "_diretores",
        F.when(
            F.col("d.diretores").isNull()
            | (
                F.length(
                    F.trim(
                        F.col("d.diretores")
                    )
                ) == 0
            ),
            F.lit("direção não informada")
        )
        .otherwise(
            F.col("d.diretores")
        )
    )

    .withColumn(
        "_sinopse",
        F.coalesce(
            F.col("m.sinopse"),
            F.lit("Sinopse não informada")
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_context_base = (
    df_context_base

    .withColumn(
        "_receita",
        F.when(
            F.col("f.receita_brl").isNotNull(),
            F.concat(
                F.lit("R$ "),
                F.format_number(
                    F.col("f.receita_brl"),
                    2
                )
            )
        )
        .otherwise(
            F.lit("receita não informada")
        )
    )

    .withColumn(
        "_orcamento",
        F.when(
            F.col("f.orcamento_brl").isNotNull(),
            F.concat(
                F.lit("R$ "),
                F.format_number(
                    F.col("f.orcamento_brl"),
                    2
                )
            )
        )
        .otherwise(
            F.lit("orçamento não informado")
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



## 13. Montagem do documento LLM


In [0]:
df_genai_context = (
    df_context_base

    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "),
            F.col("_titulo"),

            F.lit(", lançado no ano de "),
            F.col("_ano"),

            F.lit(", faturou "),
            F.col("_receita"),

            F.lit(" e teve um custo de "),
            F.col("_orcamento"),

            F.lit(". Estrelado por "),
            F.col("_atores"),

            F.lit(" e dirigido por "),
            F.col("_diretores"),

            F.lit(
                ", o filme possui a seguinte sinopse: "
            ),

            F.col("_sinopse"),
            F.lit(".")
        )
    )

    .select(
        F.col("m.id_filme")
        .alias("movie_id"),

        F.col("m.titulo")
        .alias("title"),

        "llm_context_document"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
save_gold(
    df_genai_context,
    "gold_genai_movies_context"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.gold.gold_genai_movies_context criada com sucesso.



## 14. Validações Finais

##### Star Schema

In [0]:
expected_gold_tables = [
    "dim_movies",
    "dim_genres",
    "dim_people",
    "dim_companies",
    "dim_reviews",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company",
    "fact_movies_performance",
    "gold_genai_movies_context"
]

gold_tables = [
    table.name
    for table in spark.catalog.listTables(
        f"{CATALOG}.{GOLD}"
    )
]

for table in expected_gold_tables:
    if table in gold_tables:
        count = spark.table(
            f"{CATALOG}.{GOLD}.{table}"
        ).count()

        print(
            f"OK - {table}: {count} registros"
        )

    else:
        print(
            f"ERRO - {table} não encontrada"
        )

OK - dim_movies: 97879 registros
OK - dim_genres: 19 registros
OK - dim_people: 419311 registros
OK - dim_companies: 45122 registros
OK - dim_reviews: 27303 registros
OK - bridge_movie_genre: 130994 registros
OK - bridge_movie_person: 767500 registros
OK - bridge_movie_company: 118054 registros
OK - fact_movies_performance: 96463 registros
OK - gold_genai_movies_context: 96463 registros


##### Chaves da fato


In [0]:
display(
    spark.table(
        "workspace.gold.fact_movies_performance"
    )
    .groupBy("sk_movie_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

sk_movie_id,count


##### FKs sem dimensão

In [0]:
df_fact = spark.table(
    "workspace.gold.fact_movies_performance"
)

df_movies = spark.table(
    "workspace.gold.dim_movies"
)

orfaos = (
    df_fact.alias("f")
    .join(
        df_movies.alias("m"),
        F.col("f.sk_movie_id")
        == F.col("m.sk_movie_id"),
        "left_anti"
    )
)

print(
    "FKs de filmes sem dimensão:",
    orfaos.count()
)

FKs de filmes sem dimensão: 0


## 15. Analytics

#### 15.1 Qual é a receita total em R$ somada de todos os filmes

In [0]:
display(
    spark.table(
        "workspace.gold.fact_movies_performance"
    )
    .agg(
        F.round(
            F.sum("receita_brl"),
            2
        ).alias(
            "receita_total_brl"
        )
    )
)

receita_total_brl
834762681404.23


#### 15.2 Quais são os 5 filmes com maior popularidade?

In [0]:
df_fact = spark.table(
    "workspace.gold.fact_movies_performance"
)

df_movies = spark.table(
    "workspace.gold.dim_movies"
)

top_5_popularidade = (
    df_fact.alias("f")
    .join(
        df_movies.alias("m"),
        "sk_movie_id"
    )
    .select(
        "m.titulo",
        "f.popularidade"
    )
    .filter(
        F.col("popularidade").isNotNull()
    )
    .orderBy(
        F.col("popularidade").desc()
    )
    .limit(5)
)

display(top_5_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


#### 15.3 Quantos filmes cada gênero possui?

In [0]:
df_genres = spark.table(
    "workspace.gold.dim_genres"
)

df_bridge_genre = spark.table(
    "workspace.gold.bridge_movie_genre"
)

filmes_por_genero = (
    df_bridge_genre.alias("b")
    .join(
        df_genres.alias("g"),
        "sk_genre_id"
    )
    .groupBy(
        "g.nome_genero"
    )
    .agg(
        F.countDistinct(
            "b.sk_movie_id"
        ).alias(
            "qtd_filmes"
        )
    )
    .orderBy(
        F.col("qtd_filmes").desc(),
        F.col("nome_genero")
    )
)

display(filmes_por_genero)

nome_genero,qtd_filmes
Drama,30420
Documentary,18614
Comedy,17327
Thriller,9409
Horror,9152
Romance,6983
Action,5526
Crime,4322
Animation,4155
TV Movie,3672


#### 15.4 Para os 10 filmes de maior receita, mostrar título, receita USD, receita BRL e posição usando RANK().

In [0]:
w_rank_receita = (
    Window
    .orderBy(
        F.col("receita_usd").desc()
    )
)

ranking_receita = (
    df_fact
    .filter(
        F.col("receita_usd").isNotNull()
    )
    .withColumn(
        "posicao",
        F.rank().over(w_rank_receita)
    )
    .filter(
        F.col("posicao") <= 10
    )
    .join(
        df_movies,
        "sk_movie_id"
    )
    .select(
        "posicao",
        "titulo",
        "receita_usd",
        "receita_brl"
    )
    .orderBy(
        "posicao"
    )
)

display(ranking_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


#### 15.5 Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

In [0]:
data_limite = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        &
        F.col("data_lancamento").isNotNull()
        &
        (
            F.col("data_lancamento")
            <= F.current_date()
        )
    )
    .agg(
        F.max(
            "data_lancamento"
        ).alias("data_max")
    )
    .first()["data_max"]
)

print(
    f"Data limite da análise: {data_limite}"
)

Data limite da análise: 2026-02-19


In [0]:
df_people = spark.table(
    "workspace.gold.dim_people"
)

df_bridge_people = spark.table(
    "workspace.gold.bridge_movie_person"
)

In [0]:
data_inicio_2_anos = (
    df_movies
    .select(
        F.add_months(
            F.lit(data_limite),
            -24
        ).alias("data_inicio")
    )
    .first()["data_inicio"]
)

print(
    f"Período: {data_inicio_2_anos} até {data_limite}"
)

Período: 2024-02-19 até 2026-02-19


In [0]:
participacoes_atores = (
    df_bridge_people.alias("b")

    .join(
        df_people.alias("p"),
        "sk_person_id"
    )

    .join(
        df_movies.alias("m"),
        "sk_movie_id"
    )

    .filter(
        (F.col("p.tipo_pessoa") == "Ator")
        &
        (F.col("m.status_filme") == "Lançado")
        &
        (
            F.col("m.data_lancamento")
            >= F.lit(data_inicio_2_anos)
        )
        &
        (
            F.col("m.data_lancamento")
            <= F.lit(data_limite)
        )
    )

    .groupBy(
        "p.nome_pessoa"
    )

    .agg(
        F.countDistinct(
            "sk_movie_id"
        ).alias(
            "qtd_participacoes"
        )
    )
)

In [0]:
w_rank_atores = Window.orderBy(
    F.col("qtd_participacoes").desc()
)

resultado_ator = (
    participacoes_atores
    .withColumn(
        "posicao",
        F.rank().over(
            w_rank_atores
        )
    )
    .filter(
        F.col("posicao") == 1
    )
)

display(resultado_ator)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_participacoes,posicao
Kevin Hart,64,1



#### 15.6 Qual produtora teve o maior lucro nos últimos 5 anos?


In [0]:
df_companies = spark.table(
    "workspace.gold.dim_companies"
)

df_bridge_company = spark.table(
    "workspace.gold.bridge_movie_company"
)

In [0]:
data_inicio_5_anos = (
    df_movies
    .select(
        F.add_months(
            F.lit(data_limite),
            -60
        ).alias("data_inicio")
    )
    .first()["data_inicio"]
)

print(
    f"Período: {data_inicio_5_anos} até {data_limite}"
)

Período: 2021-02-19 até 2026-02-19


In [0]:
lucro_produtoras = (
    df_bridge_company.alias("b")

    .join(
        df_companies.alias("c"),
        "sk_company_id"
    )

    .join(
        df_movies.alias("m"),
        "sk_movie_id"
    )

    .join(
        df_fact.alias("f"),
        "sk_movie_id"
    )

    .filter(
        (
            F.col("m.data_lancamento")
            >= F.lit(data_inicio_5_anos)
        )
        &
        (
            F.col("m.data_lancamento")
            <= F.lit(data_limite)
        )
    )

    .groupBy(
        "c.nome_produtora"
    )

    .agg(
        F.round(
            F.sum(
                "f.lucro_brl"
            ),
            2
        ).alias(
            "lucro_total_brl"
        )
    )
)

In [0]:
w_rank_produtoras = (
    Window
    .orderBy(
        F.col("lucro_total_brl").desc()
    )
)

resultado_produtora = (
    lucro_produtoras
    .withColumn(
        "posicao",
        F.rank().over(
            w_rank_produtoras
        )
    )
    .filter(
        F.col("posicao") == 1
    )
)

display(resultado_produtora)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_produtora,lucro_total_brl,posicao
Universal Pictures,29767326921.64,1
